# Ch 14　長期記憶與跨對話 store

> **本 notebook 完全不需要 API key。**

In [ ]:
!uv pip install -q langgraph

## 14.2　store 基礎：put 與 search（記憶用 namespace 分類）

In [ ]:
import uuid
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
user_id = "1"
namespace = (user_id, "memories")     # namespace 是個 tuple，可任意設計

# put：存一筆記憶（namespace + 唯一 key + 值 dict）
store.put(namespace, str(uuid.uuid4()), {"food_preference": "我喜歡披薩"})
store.put(namespace, str(uuid.uuid4()), {"hobby": "爬山"})

# search：讀出某 namespace 下的記憶
for item in store.search(namespace):
    print("  ", item.value, "| key:", item.key[:8], "| ns:", item.namespace)

## 14.1　長期 vs 短期：store 跨 thread，checkpointer 不行

store 與 thread 無關——同一個 store，任何對話（thread）都讀得到。這正是「跨對話記住使用者」的關鍵。

In [ ]:
# 模擬：對話 A 把偏好存進 store
store.put(("user-42", "memories"), "pref-1", {"likes": "深色模式"})

# 對話 B（完全不同的 thread / session）一樣讀得到 —— 因為 store 不綁 thread
got = store.search(("user-42", "memories"))
print("另一段對話也讀得到：", [i.value for i in got])

## 14.3　namespace 是「前綴比對」

In [ ]:
store.put(("alice", "memories"), "m1", {"note": "記憶一則"})
store.put(("alice", "preferences"), "p1", {"theme": "dark"})

# 用上層前綴 ("alice",) 搜尋 → 連底下的 memories 與 preferences 都撈到
print("前綴 ('alice',) 撈到：")
for item in store.search(("alice",)):
    print("  ", item.namespace, item.value)
print("=> 要精確就傳完整 namespace；要跨類就用前綴。")